# Idealized Synthetic Data

*Under development*

In [1]:
import numpy as np
from IPython.display import display  # so can run as script too

from melodies_monet import driver
from melodies_monet.tutorial import model, pt_sfc_obs, swath_grid_2d

In [ ]:
an = driver.analysis()
an.control = "control_idealized.yaml"
an.read_control()
an

````{admonition} Note: This is the complete file that was loaded.
:class: dropdown

```{literalinclude} control_idealized.yaml
:caption:
:linenos:
```
````

## Generate data

### Model

In [ ]:
control = an.control_dict

ds_mod = model(control, freq="3h")
ds_mod

In [ ]:
ds_mod.squeeze("z").A.plot(col="time", col_wrap=5, size=3);

In [5]:
ds_mod.to_netcdf(control['model']['idealized']['files'])

### Obs

In [ ]:
ds = pt_sfc_obs(control, model=ds_mod)
ds

In [7]:
ds.to_netcdf(control['obs']['test_obs']['filename'])

### Sat

In [ ]:
# Generate swath grid (TEMPO-like)
sat_lon = -91
g = swath_grid_2d(
    0, sat_lon, 35_786,
    view_cross=(-8, 18, 30),
    view_along=(-8, 8, 20),
    look_angle=0,
    look_azimuth=1.23,
)
display(g)

lats, lons = g.lat.values, g.lon.values

# Print first few points
print(f"{lats.size} points {np.isnan(lons).sum()/lons.size:.2%} null")
print("First 5 non-null swath points:")
i, n = -1, 0
while n < 5:
    i += 1
    if np.isnan(lons.flat[i]):
        continue
    print(f"Point {i+1}: {lats.flat[i]:.4f}°, {lons.flat[i]:.4f}°")
    n += 1

import cartopy.crs as ccrs
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(lons, lats)

fig = plt.figure()
ax = fig.add_subplot(projection=ccrs.Geostationary(central_longitude=sat_lon))
ax.coastlines()
ax.scatter(lons, lats, transform=ccrs.PlateCarree())

# Also plot opposite side of Earth just to make sure no points are there
fig = plt.figure()
ax = fig.add_subplot(projection=ccrs.Geostationary(central_longitude=(sat_lon + 180) % 360))
ax.coastlines()
ax.scatter(lons, lats, transform=ccrs.PlateCarree())



## Load

In [ ]:
an.open_models()

In [ ]:
an.models['idealized'].obj

In [11]:
an.open_obs()

In [ ]:
an.obs['test_obs'].obj

## Pair

In [ ]:
%%time

an.pair_data()

In [ ]:
an.paired

In [ ]:
an.paired['test_obs_idealized'].obj

In [ ]:
an.paired['test_obs_idealized'].obj.dims

## Plot

In [ ]:
%%time

an.plotting()

## Save/load paired data -- netCDF

And compare to the original pair object.

In [ ]:
from copy import deepcopy

p0 = deepcopy(an.paired['test_obs_idealized'].obj)

an.save_analysis()
an.read_analysis()
p1 = deepcopy(an.paired['test_obs_idealized'].obj)
p1.close()

display(p0)
display(p1)
assert p1 is not p0 and p1.equals(p0)

## Save/load paired data -- Python object

In [ ]:
print(an.save)
an.save["paired"]["method"] = "pkl"
del an.save["paired"]["prefix"]
# We could leave `prefix` since unused, but we need to set `output_name`
an.save["paired"]["output_name"] = "asdf.joblib"
print("->", an.save)

print()
print(an.read)
an.read["paired"]["method"] = "pkl"
an.read["paired"]["filenames"] = "asdf.joblib"
print("->", an.read)

print()
an.save_analysis()
an.read_analysis()
p2 = deepcopy(an.paired['test_obs_idealized'].obj)
p2.close()

# display(p0)
display(p2)
assert p2 is not p0 and p2.equals(p0)